Git clone

In [ ]:
!git clone https://github.com/nicolo-monzu/federated-learning.git
%cd /content/federated-learning

Mount and link Google Drive (optional)

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/centralized_model/checkpoints"
repo_folder = "/content/federated-learning/centralized_model/checkpoints"

os.makedirs(drive_folder, exist_ok=True)
os.makedirs(repo_folder, exist_ok=True)
shutil.rmtree(repo_folder)

os.symlink(drive_folder, repo_folder)

drive_folder_federated = "/content/drive/MyDrive/federated_model/checkpoints"
repo_folder_federated = "/content/federated-learning/federated_model/checkpoints"

os.makedirs(drive_folder_federated, exist_ok=True)
os.makedirs(repo_folder_federated, exist_ok=True)
shutil.rmtree(repo_folder_federated)

os.symlink(drive_folder_federated, repo_folder_federated)

Download dataset from drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

!mkdir -p /content/federated-learning/dataset/
!cp /content/drive/MyDrive/cifar-100-python.tar.gz /content/federated-learning/dataset/
!tar -xzvf dataset/cifar-100-python.tar.gz

Train

In [ ]:
# !python train.py start -h
!python train.py start

Train Federated

In [ ]:
!python train_federated.py start

Hyperparameter search

In [ ]:
!python hp_search.py

Federated learning experiments

In [ ]:
from train_federated import start

J = 4
Nc = 100

start(run_name=f"federated_j_{J}_nc_{Nc}",
      num_rounds = 352*4 // J,
      num_steps_per_client = J,
      num_classes_per_client = Nc,
      rounds_per_scheduler_step = 16 // J,
      scale_grow_interval = 48*4 // J,
      validation_interval = 64 // J
      )

# Copy logs and plots into Google Drive after the run has finished
drive_root = "/content/drive/MyDrive"

folders_to_copy = ["logs", "plots"]

if os.path.isdir(drive_root):
    for folder in folders_to_copy:
        source_folder = f"/content/federated-learning/federated_model/{folder}"
        drive_folder = f"/content/drive/MyDrive/federated_model/{folder}"

        if os.path.isdir(source_folder):
            os.makedirs(drive_folder, exist_ok=True)
            shutil.copytree(source_folder, drive_folder,dirs_exist_ok=True)
            print(f"{folder} copied to: {drive_folder}")
        else:
            print(f"{folder} folder not found: {source_folder}")
else:
    print("Google Drive is not mounted. Plots and logs were not copied.")

See transformed data

In [ ]:
%run utils/dataviewer.py